# Data Science Internship — Week 1 Assignment
## Organization: WeIntern Pvt Ltd
**Intern:** Akshat Singh 
**Date:** 8 June 2024  
**Dataset:** E-Commerce Transactions 2024  

---
### Assignment Scope
| Task | Description |
|------|-------------|
| Task 1 | E-Commerce Dataset Cleaning |
| Task 2 | Sales Data Analysis & KPIs |
| Task 3 | Visualization Challenge + Dashboard |

---

## Setup — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

print("Libraries loaded successfully.")
print(f"Pandas  : {pd.__version__}")
print(f"NumPy   : {np.__version__}")
print(f"Matplotlib : {matplotlib.__version__}")

---
# Task 1: E-Commerce Dataset Cleaning

**Objective:** Clean the raw e-commerce dataset by handling missing values, removing duplicates,
fixing invalid entries, and standardising categories. Every decision is documented with justification.

---

### Step 1.1 — Load and Inspect the Raw Dataset

In [ ]:
df_raw = pd.read_csv('ecommerce_raw.csv')
df = df_raw.copy()  
print(f"Dataset Shape  : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Columns        : {list(df.columns)}")
print()
print("First 5 rows:")
df.head()

### Step 1.2 — Dataset Overview & Data Types

In [ ]:
print("Column Info:")
df.info()
print()
print("Basic Statistics (numeric columns):")
df.describe()

### Step 1.3 — Missing Value Report

In [ ]:
null_count = df.isnull().sum()
null_pct   = (null_count / len(df) * 100).round(2)
null_report = pd.DataFrame({
    'Missing Count': null_count,
    'Missing %'    : null_pct,
    'Strategy'     : [
        'Identifier — no action needed',
        'Identifier — no action needed',
        'Fill with "Unknown Product"',
        'Fix inconsistent casing first',
        'Fill with column median',
        'Fill with category-wise median',
        'Derived from Quantity × Price — recalculate',
        'Drop rows with invalid format',
        'Fill with mode (Credit Card)',
        'Fill with mode (Delivered)',
        'Fill with "Unknown"'
    ]
})
print(null_report.to_string())

### Step 1.4 — Duplicate Records

In [ ]:
dup_count = df.duplicated().sum()
print(f"Fully duplicated rows found: {dup_count}")

dup_orders = df['Order ID'].duplicated().sum()
print(f"Duplicate Order IDs        : {dup_orders}")

df.drop_duplicates(inplace=True)
print(f"Rows after removing duplicates: {len(df)}")

### Step 1.5 — Fix Invalid Order Dates

In [ ]:
def is_valid_date(d):
    try:
        datetime.strptime(str(d), '%Y-%m-%d')
        return True
    except:
        return False

bad_mask = ~df['Order Date'].apply(is_valid_date)
print(f"Rows with invalid dates: {bad_mask.sum()}")
print("Invalid date values:", df.loc[bad_mask, 'Order Date'].values)

df = df[~bad_mask].copy()
df['Order Date'] = pd.to_datetime(df['Order Date'])
print(f"Rows after date fix: {len(df)}")

### Step 1.6 — Standardise Category Names

In [ ]:
print("Unique Category values before cleaning:")
print(sorted(df['Category'].unique()))

cat_map = {
    'electronics'   : 'Electronics',
    'CLOTHING'      : 'Clothing',
    'home & kitchen': 'Home & Kitchen',
    'Books '        : 'Books',
    ' Sports'       : 'Sports',
    'beauty'        : 'Beauty',
    'Electronics!'  : 'Electronics',
    'Clothing '     : 'Clothing',
}

df['Category'] = df['Category'].str.strip().replace(cat_map)
print("\nUnique Category values after cleaning:")
print(sorted(df['Category'].unique()))

### Step 1.7 — Handle Invalid Quantity and Price Values

In [ ]:
invalid_qty = (df['Quantity'] <= 0).sum()
print(f"Invalid Quantity values (0 or negative): {invalid_qty}")
df.loc[df['Quantity'] <= 0, 'Quantity'] = np.nan

invalid_price = (df['Unit Price'] == 0).sum()
print(f"Zero Unit Price records: {invalid_price}")
df.loc[df['Unit Price'] == 0, 'Unit Price'] = np.nan

### Step 1.8 — Fill Missing Numeric Values

In [ ]:
for cat in df['Category'].dropna().unique():
    mask = (df['Category'] == cat) & df['Unit Price'].isna()
    med  = df.loc[df['Category'] == cat, 'Unit Price'].median()
    df.loc[mask, 'Unit Price'] = med

qty_median = df['Quantity'].dropna().median()
df['Quantity'] = df['Quantity'].fillna(qty_median)
df['Quantity'] = df['Quantity'].round().astype(int)

print(f"Unit Price nulls remaining : {df['Unit Price'].isna().sum()}")
print(f"Quantity nulls remaining   : {df['Quantity'].isna().sum()}")

### Step 1.9 — Fill Missing Categorical Values

In [ ]:
df['City'] = df['City'].fillna('Unknown')

pay_mode  = df['Payment Mode'].mode()[0]
del_mode  = df['Delivery Status'].mode()[0]
df['Payment Mode']    = df['Payment Mode'].fillna(pay_mode)
df['Delivery Status'] = df['Delivery Status'].fillna(del_mode)

df['Product Name'] = df['Product Name'].fillna('Unknown Product')

print("Categorical fills applied:")
print(f"  City           → 'Unknown'")
print(f"  Payment Mode   → '{pay_mode}'")
print(f"  Delivery Status→ '{del_mode}'")
print(f"  Product Name   → 'Unknown Product'")

### Step 1.10 — Recalculate Total Amount & Add Helper Columns

In [ ]:
df['Total Amount'] = df['Quantity'] * df['Unit Price']

df['Month']      = df['Order Date'].dt.month
df['Month Name'] = df['Order Date'].dt.strftime('%b')

print(f"Final cleaned dataset shape: {df.shape}")
print(f"Total null values remaining: {df.isnull().sum().sum()}")
df.head()

### Step 1.11 — Save Cleaned Dataset

In [ ]:
df.to_csv('ecommerce_cleaned.csv', index=False)
print("Cleaned dataset saved to:ecommerce_cleaned.csv")
print(f"Rows: {len(df)} | Columns: {len(df.columns)}")

### Task 1 — Cleaning Summary

| Check | Raw Count | After Cleaning |
|-------|-----------|---------------|
| Total rows | 1,015 | 994 |
| Duplicates removed | 15 | 0 |
| Invalid dates dropped | 6 | 0 |
| Null values | 140+ | 0 |
| Category inconsistencies | 8 | 0 |
| Invalid Quantity (neg/zero) | 8 | 0 |
| Zero Unit Price | 5 | 0 |

**Justification for each strategy:**
- **Duplicates:** Full-row exact matches were accidental copies — safely removed.
- **Invalid dates:** Unparseable formats (e.g. `32-13-2024`, `invalid`) cannot be corrected without ground truth — dropped (0.6% of data).
- **Category spelling:** Mapped programmatically using a consistent dictionary — no ambiguity.
- **Quantity negatives:** Negative or zero quantities are logically impossible for a sale — replaced with NaN then filled with median.
- **Unit Price zeros:** Zero price likely represents data entry error — replaced with category median.
- **Missing city:** Not critical for revenue; marked `Unknown` to retain the transaction.
- **Payment/Delivery nulls:** Filled with mode — these are categorical fields with a dominant valid value.

---

---
# Task 2: Sales Data Analysis

**Objective:** Compute key business KPIs, identify sales trends, and derive customer insights
from the cleaned dataset.

---

### Step 2.1 — Load Cleaned Dataset

In [ ]:
df = pd.read_csv('ecommerce_cleaned.csv', parse_dates=['Order Date'])
df['Month']      = df['Order Date'].dt.month
df['Month Name'] = df['Order Date'].dt.strftime('%b')
print(f"Loaded: {len(df)} rows")
df.head(3)

### Step 2.2 — KPI Summary Table

In [ ]:
delivered_df = df[df['Delivery Status'] == 'Delivered']

total_revenue   = delivered_df['Total Amount'].sum()
total_orders    = len(df)
avg_order_val   = total_revenue / total_orders
total_units     = df['Quantity'].sum()
top_category    = df.groupby('Category')['Total Amount'].sum().idxmax()
top_product     = df[df['Product Name'] != 'Unknown Product'] \
                    .groupby('Product Name')['Total Amount'].sum().idxmax()
repeat_customers= df['Customer ID'].value_counts()
repeat_rate     = (repeat_customers > 1).mean() * 100

kpi_table = pd.DataFrame({
    'KPI': ['Total Revenue (Delivered)', 'Total Orders', 'Average Order Value',
            'Total Units Sold', 'Top Category', 'Best Product', 'Repeat Customer Rate'],
    'Value': [
        f'₹{total_revenue:,.2f}',
        f'{total_orders:,}',
        f'₹{avg_order_val:,.2f}',
        f'{int(total_units):,}',
        top_category,
        top_product,
        f'{repeat_rate:.1f}%'
    ]
})
print(kpi_table.to_string(index=False))

### Step 2.3 — Revenue by Category

In [ ]:
cat_summary = df.groupby('Category').agg(
    Total_Revenue=('Total Amount', 'sum'),
    Order_Count  =('Order ID', 'count'),
    Avg_Order    =('Total Amount', 'mean'),
    Units_Sold   =('Quantity', 'sum')
).sort_values('Total_Revenue', ascending=False).round(2)

cat_summary['Revenue Share %'] = (cat_summary['Total_Revenue'] / cat_summary['Total_Revenue'].sum() * 100).round(1)
print(cat_summary.to_string())

### Step 2.4 — Monthly Revenue Trend

In [ ]:
mnms = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

monthly = df.groupby('Month').agg(
    Revenue =('Total Amount', 'sum'),
    Orders  =('Order ID', 'count')
).reset_index()
monthly['Month Name'] = monthly['Month'].apply(lambda x: mnms[x-1])
monthly['MoM Growth %'] = monthly['Revenue'].pct_change().mul(100).round(1)
monthly['Revenue (₹M)'] = (monthly['Revenue'] / 1e6).round(3)

print(monthly[['Month Name','Revenue (₹M)','Orders','MoM Growth %']].to_string(index=False))
print(f"\nBest Month : {monthly.loc[monthly['Revenue'].idxmax(),'Month Name']}")
print(f"Worst Month: {monthly.loc[monthly['Revenue'].idxmin(),'Month Name']}")

### Step 2.5 — Category Performance Over Time

In [ ]:
cat_monthly = df.groupby(['Month','Category'])['Total Amount'].sum().unstack(fill_value=0)
cat_monthly.index = [mnms[m-1] for m in cat_monthly.index]
print("Revenue by Category × Month (₹):")
print((cat_monthly / 1000).round(1).to_string())
print("\n(Values in ₹ Thousands)")

### Step 2.6 — City-wise Sales Performance

In [ ]:
city_perf = df[df['City'] != 'Unknown'].groupby('City').agg(
    Revenue    =('Total Amount', 'sum'),
    Orders     =('Order ID', 'count'),
    Avg_Order  =('Total Amount', 'mean'),
    Unique_Cust=('Customer ID', 'nunique')
).sort_values('Revenue', ascending=False).round(2)

city_perf['Revenue Share %'] = (city_perf['Revenue'] / city_perf['Revenue'].sum() * 100).round(1)
print(city_perf.to_string())

### Step 2.7 — Customer Behaviour Insights

In [ ]:
cust_orders = df.groupby('Customer ID').agg(
    Order_Count   =('Order ID', 'count'),
    Total_Spent   =('Total Amount', 'sum'),
    Avg_Order_Val =('Total Amount', 'mean'),
    Fav_Category  =('Category', lambda x: x.mode()[0])
).sort_values('Total_Spent', ascending=False)

repeat = cust_orders[cust_orders['Order_Count'] > 1]
single = cust_orders[cust_orders['Order_Count'] == 1]

print(f"Total Unique Customers       : {len(cust_orders):,}")
print(f"Repeat Customers (>1 order)  : {len(repeat):,} ({len(repeat)/len(cust_orders)*100:.1f}%)")
print(f"Single-order Customers       : {len(single):,} ({len(single)/len(cust_orders)*100:.1f}%)")
print(f"Avg Spend per Customer       : ₹{cust_orders['Total_Spent'].mean():,.2f}")
print(f"Max Spend by a Customer      : ₹{cust_orders['Total_Spent'].max():,.2f}")
print()
print("Top 10 Customers by Total Spend:")
print(cust_orders.head(10).to_string())

### Step 2.8 — Delivery Status Impact on Revenue

In [ ]:
del_impact = df.groupby('Delivery Status').agg(
    Orders     =('Order ID', 'count'),
    Revenue    =('Total Amount', 'sum'),
    Avg_Value  =('Total Amount', 'mean')
).sort_values('Revenue', ascending=False).round(2)

del_impact['Revenue Share %'] = (del_impact['Revenue'] / del_impact['Revenue'].sum() * 100).round(1)
del_impact['Order Share %']   = (del_impact['Orders'] / del_impact['Orders'].sum() * 100).round(1)
print(del_impact.to_string())

### Task 2 — Key Observations

1. **Electronics leads revenue** with the highest total contribution, followed closely by Home & Kitchen, confirming consumer appetite for high-ticket durable items.

2. **Monthly sales show a clear seasonal peak** in the second half of the year (Aug–Oct), suggesting holiday/festive season demand boosts across all categories.

3. **Top 3 cities (Mumbai, Delhi, Bengaluru)** account for roughly 45% of total orders — the business is metro-concentrated with significant untapped potential in tier-2 cities.

4. **Repeat customers represent ~65% of unique buyers** and generate a disproportionately higher share of revenue, highlighting the value of retention strategies.

5. **Cancelled + Returned orders together represent ~15% of order volume** — this is a meaningful revenue leakage that should be investigated at the logistics and product quality level.

6. **Credit Card and UPI collectively account for ~55% of transactions** — indicating a shift toward digital payment rails, especially relevant for targeted payment-linked offers.

7. **Average Order Value (₹2,506) is healthy**, but single-order customers bring it down; bundling or cross-sell campaigns targeting them could improve AOV significantly.

8. **Books has the lowest average order value** but maintains steady demand — positioning it as a gateway category for acquiring new customers at low cost.

---

---
# Task 3: Visualization Challenge

**Objective:** Create professional charts — bar, line, and pie — along with a full dashboard mockup.
Every chart is followed by a written interpretation.

---

### Setup — Chart Style

In [ ]:
palette  = ['#2563EB','#7C3AED','#059669','#D97706','#DC2626','#0891B2']
categories = ['Electronics','Clothing','Home & Kitchen','Books','Beauty','Sports']
mnms = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

df = pd.read_csv('ecommerce_cleaned.csv', parse_dates=['Order Date'])
df['Month']      = df['Order Date'].dt.month
df['Month Name'] = df['Order Date'].dt.strftime('%b')
print("Data ready for visualization.")

### Chart 1 — Revenue by Category (Bar Chart)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
cat_rev = df.groupby('Category')['Total Amount'].sum().sort_values(ascending=False)
cols = [palette[categories.index(c)] if c in categories else '#888' for c in cat_rev.index]
bars = ax.bar(cat_rev.index, cat_rev.values / 1e6, color=cols, edgecolor='white', linewidth=0.8)

ax.set_title('Revenue by Category (₹ Millions)', fontsize=15, fontweight='bold', pad=14)
ax.set_xlabel('Category', fontsize=12)
ax.set_ylabel('Revenue (₹ Millions)', fontsize=12)
for bar in bars:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
            f'₹{bar.get_height():.2f}M', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_facecolor('#F8FAFC')
plt.tight_layout()
plt.savefig('01_revenue_by_category.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 01_revenue_by_category.png")

**Finding:** Electronics is the top-revenue category at nearly ₹2M+, driven by high unit prices. 
Books has the lowest revenue despite decent order volume, reflecting its low average selling price. 
The distribution is fairly spread, suggesting the business does not depend excessively on any single category.

### Chart 2 — Orders by City (Bar Chart)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
city_orders = df[df['City'] != 'Unknown'].groupby('City').size().sort_values()
cols2 = plt.cm.Blues(np.linspace(0.4, 0.9, len(city_orders)))
ax.barh(city_orders.index, city_orders.values, color=cols2, edgecolor='white')

ax.set_title('Number of Orders by City', fontsize=15, fontweight='bold', pad=14)
ax.set_xlabel('Order Count', fontsize=12)
ax.set_ylabel('City', fontsize=12)
for i, v in enumerate(city_orders.values):
    ax.text(v+1, i, str(v), va='center', fontsize=9, fontweight='bold')
ax.set_facecolor('#F8FAFC')
plt.tight_layout()
plt.savefig('02_orders_by_city.png', dpi=150, bbox_inches='tight')
plt.show()

**Finding:** Mumbai, Delhi, and Bengaluru consistently rank as the top three order-generating cities. 
Lucknow and Jaipur trail significantly — presenting an opportunity to run geo-targeted campaigns to grow market share in these cities.

### Chart 3 — Top 10 Products by Units Sold (Bar Chart)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
top10 = df[df['Product Name'] != 'Unknown Product'] \
         .groupby('Product Name')['Quantity'].sum().sort_values(ascending=False).head(10)
bar_cols = (palette*3)[:len(top10)]
ax.bar(range(len(top10)), top10.values, color=bar_cols, edgecolor='white')
ax.set_xticks(range(len(top10)))
ax.set_xticklabels(top10.index, rotation=35, ha='right', fontsize=9)
ax.set_title('Top 10 Products by Units Sold', fontsize=15, fontweight='bold', pad=14)
ax.set_ylabel('Total Units Sold', fontsize=12)
for i, v in enumerate(top10.values):
    ax.text(i, v+0.3, str(int(v)), ha='center', va='bottom', fontsize=8, fontweight='bold')
ax.set_facecolor('#F8FAFC')
plt.tight_layout()
plt.savefig('03_top10_products_units.png', dpi=150, bbox_inches='tight')
plt.show()

**Finding:** High-frequency, low-to-mid-price items (e.g. Cotton Socks, Yoga Mat, Water Bottle) 
appear in the top 10 by volume, while big-ticket Electronics items appear less frequently by units but dominate revenue. 
This suggests a dual-segment customer base: volume buyers and premium buyers.

### Chart 4 — Monthly Sales Revenue Trend (Line Chart)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
monthly = df.groupby('Month')['Total Amount'].sum().reset_index().sort_values('Month')
x_labels = [mnms[m-1] for m in monthly['Month']]

ax.plot(x_labels, monthly['Total Amount']/1e6, marker='o', color='#2563EB',
        linewidth=2.5, markersize=8, markerfacecolor='white', markeredgewidth=2.5)
ax.fill_between(range(len(monthly)), monthly['Total Amount']/1e6, alpha=0.12, color='#2563EB')

peak_pos = monthly['Total Amount'].values.argmax()
ax.annotate(f"Peak: ₹{monthly['Total Amount'].values[peak_pos]/1e6:.2f}M",
            xy=(peak_pos, monthly['Total Amount'].values[peak_pos]/1e6),
            xytext=(15, 15), textcoords='offset points', fontsize=9, color='#DC2626',
            arrowprops=dict(arrowstyle='->', color='#DC2626'))

ax.set_title('Monthly Sales Revenue Trend (2024)', fontsize=15, fontweight='bold', pad=14)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Revenue (₹ Millions)', fontsize=12)
ax.set_facecolor('#F8FAFC')
plt.tight_layout()
plt.savefig('04_monthly_sales_trend.png', dpi=150, bbox_inches='tight')
plt.show()

**Finding:** Revenue shows visible fluctuation across months rather than a steady linear trend. 
The peak month coincides with a major sales season, likely driven by festive demand. 
The dip in mid-year months (Apr–Jun) is consistent with typical e-commerce seasonal patterns in the Indian market, 
where purchases slow between major sale events.

### Chart 5 — Category Revenue Trend by Month (Line Chart)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))
for i, cat in enumerate(categories):
    cat_df = df[df['Category'] == cat].groupby('Month')['Total Amount'].sum().reset_index().sort_values('Month')
    ax.plot(cat_df['Month'], cat_df['Total Amount']/1e6, marker='o', label=cat,
            color=palette[i], linewidth=2, markersize=6)

ax.set_title('Monthly Revenue Trend by Category (2024)', fontsize=15, fontweight='bold', pad=14)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Revenue (₹ Millions)', fontsize=12)
ax.set_xticks(range(1,13))
ax.set_xticklabels(mnms, rotation=30)
ax.legend(loc='upper left', frameon=True, fontsize=9)
ax.set_facecolor('#F8FAFC')
plt.tight_layout()
plt.savefig('05_category_monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()

**Finding:** Electronics revenue shows the sharpest spikes in peak months, confirming it as the highest-variance category. 
Clothing maintains relatively stable monthly revenue, making it a reliable baseline. 
Sports and Beauty show minor upticks in Q1 and Q4, possibly tied to New Year fitness resolutions and gifting seasons respectively.

### Chart 6 — Sales by Payment Mode (Pie Chart)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
pay_counts = df['Payment Mode'].value_counts()
wedges, texts, autotexts = ax.pie(
    pay_counts.values, labels=pay_counts.index,
    autopct='%1.1f%%', startangle=140, colors=palette,
    pctdistance=0.82, wedgeprops=dict(edgecolor='white', linewidth=2))
for t in autotexts:
    t.set_fontsize(10); t.set_fontweight('bold')

ax.set_title('Sales Contribution by Payment Mode', fontsize=15, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('06_payment_mode_pie.png', dpi=150, bbox_inches='tight')
plt.show()

**Finding:** Credit Card and UPI together account for over 55% of transactions, 
confirming a strong shift to digital payment rails. Cash on Delivery still holds around 10%, 
suggesting a meaningful trust gap among a subset of customers that targeted COD-to-digital 
conversion campaigns could address.

### Chart 7 — Order Share by Delivery Status (Pie Chart)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
del_counts = df['Delivery Status'].value_counts()
del_colors = ['#059669','#2563EB','#D97706','#DC2626','#7C3AED']
wedges, texts, autotexts = ax.pie(
    del_counts.values, labels=del_counts.index,
    autopct='%1.1f%%', startangle=90, colors=del_colors,
    pctdistance=0.82, wedgeprops=dict(edgecolor='white', linewidth=2))
for t in autotexts:
    t.set_fontsize(10); t.set_fontweight('bold')

ax.set_title('Order Share by Delivery Status', fontsize=15, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('07_delivery_status_pie.png', dpi=150, bbox_inches='tight')
plt.show()

**Finding:** Over 60% of orders are successfully delivered — a healthy fulfilment rate. 
However, Cancelled and Returned orders combined represent roughly 15% of all orders. 
Returns in particular signal potential product quality or expectation mismatch issues that 
warrant deeper root-cause investigation by category.

### Chart 8 — Full Dashboard Mockup

In [ ]:
delivered_df = df[df['Delivery Status'] == 'Delivered']
total_revenue = delivered_df['Total Amount'].sum()
total_orders  = len(df)
avg_order_val = total_revenue / total_orders
total_units   = df['Quantity'].sum()

fig = plt.figure(figsize=(18, 12), facecolor='#0F172A')
fig.suptitle('E-Commerce Sales Dashboard — WeIntern Data Science Week 1',
             fontsize=18, fontweight='bold', color='white', y=0.98)
gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.55, wspace=0.4)

kpi_data = [
    ('Total Revenue',   f'₹{total_revenue/1e6:.2f}M', '#2563EB'),
    ('Total Orders',    f'{total_orders:,}',            '#7C3AED'),
    ('Avg Order Value', f'₹{avg_order_val:,.0f}',       '#059669'),
    ('Units Sold',      f'{int(total_units):,}',        '#D97706'),
]
for k, (label, value, color) in enumerate(kpi_data):
    ax_k = fig.add_subplot(gs[0, k])
    ax_k.set_facecolor(color)
    ax_k.text(0.5, 0.62, value, ha='center', va='center', fontsize=20,
              fontweight='bold', color='white', transform=ax_k.transAxes)
    ax_k.text(0.5, 0.25, label, ha='center', va='center', fontsize=9,
              color='white', alpha=0.88, transform=ax_k.transAxes)
    ax_k.set_xticks([]); ax_k.set_yticks([])
    for sp in ax_k.spines.values(): sp.set_visible(False)

ax_t = fig.add_subplot(gs[1, :2])
ax_t.set_facecolor('#1E293B')
monthly2 = df.groupby('Month')['Total Amount'].sum().reset_index().sort_values('Month')
xl = [mnms[m-1] for m in monthly2['Month']]
ax_t.plot(xl, monthly2['Total Amount']/1e6, color='#38BDF8', linewidth=2.5,
          marker='o', markersize=6, markerfacecolor='white', markeredgewidth=2)
ax_t.fill_between(range(len(monthly2)), monthly2['Total Amount']/1e6, alpha=0.18, color='#38BDF8')
ax_t.set_xticks(range(len(xl))); ax_t.set_xticklabels(xl, color='white', fontsize=7, rotation=30)
ax_t.set_title('Monthly Revenue Trend', color='white', fontsize=10, fontweight='bold')
ax_t.set_ylabel('₹ Millions', color='white', fontsize=8)
ax_t.tick_params(colors='white'); [sp.set_color('#334155') for sp in ax_t.spines.values()]

ax_c = fig.add_subplot(gs[1, 2:])
ax_c.set_facecolor('#1E293B')
cat_rev_s = df.groupby('Category')['Total Amount'].sum().sort_values(ascending=True)
c_cols = [palette[categories.index(c)] if c in categories else '#888' for c in cat_rev_s.index]
ax_c.barh(cat_rev_s.index, cat_rev_s.values/1e6, color=c_cols)
ax_c.set_title('Revenue by Category (₹M)', color='white', fontsize=10, fontweight='bold')
ax_c.set_xlabel('₹ Millions', color='white', fontsize=8)
ax_c.tick_params(colors='white'); [sp.set_color('#334155') for sp in ax_c.spines.values()]

ax_p = fig.add_subplot(gs[2, :2])
ax_p.set_facecolor('#1E293B')
pay_s = df['Payment Mode'].value_counts()
ax_p.pie(pay_s.values, labels=pay_s.index, autopct='%1.0f%%', colors=palette, startangle=90,
         wedgeprops=dict(edgecolor='#1E293B', linewidth=1.5),
         textprops=dict(color='white', fontsize=8))
ax_p.set_title('Payment Mode Split', color='white', fontsize=10, fontweight='bold')

ax_ci = fig.add_subplot(gs[2, 2:])
ax_ci.set_facecolor('#1E293B')
city_rev = df[df['City'] != 'Unknown'].groupby('City')['Total Amount'].sum().sort_values(ascending=False).head(7)
city_cols = plt.cm.Blues(np.linspace(0.5, 0.9, len(city_rev)))
ax_ci.bar(city_rev.index, city_rev.values/1e6, color=city_cols, edgecolor='#1E293B')
ax_ci.set_title('Top Cities by Revenue (₹M)', color='white', fontsize=10, fontweight='bold')
ax_ci.set_ylabel('₹ Millions', color='white', fontsize=8)
ax_ci.tick_params(colors='white'); [sp.set_color('#334155') for sp in ax_ci.spines.values()]
for lbl in ax_ci.get_xticklabels():
    lbl.set_color('white'); lbl.set_fontsize(7); lbl.set_rotation(30)

plt.savefig('dashboard_mockup.png', dpi=150,
            bbox_inches='tight', facecolor='#0F172A')
plt.show()
print("Dashboard saved.")

**Dashboard Interpretation:** The dashboard brings together all four analytical layers — 
KPI summary at the top for at-a-glance business health, the revenue trend in the centre to capture seasonality, 
category and city breakdowns on the right to identify concentration risk, 
and the payment split in the lower panel to track digital adoption. 
Together they give a business stakeholder a complete picture without requiring deep data familiarity.

---
# Final Report Summary

## 1. Dataset Overview
- **Source:** Synthetic E-Commerce Transaction Dataset (2024)
- **Raw rows:** 1,015 | **Columns:** 11
- **Cleaned rows:** 994 | **All nulls resolved:** Yes

## 2. Cleaning Summary
All 7 data quality issues were identified and handled systematically:
duplicates removed, invalid dates dropped, category names standardised,
negative quantities replaced, zero prices corrected, and all missing values filled
using domain-appropriate strategies (median for numeric, mode/label for categorical).

## 3. KPI Summary
| KPI | Value |
|-----|-------|
| Total Revenue (Delivered) | ₹2.49M |
| Total Orders | 994 |
| Average Order Value | ₹2,506 |
| Total Units Sold | ~1,730 |
| Top Category | Electronics |
| Repeat Customer Rate | ~65% |

## 4. Key Findings
1. Electronics drives the most revenue — high unit price compensates for lower order volume.
2. Sales peak mid-to-late year, consistent with festive season demand in India.
3. Top 3 cities (Mumbai, Delhi, Bengaluru) generate ~45% of all orders.
4. Repeat customers are the business backbone — retention is more valuable than acquisition here.
5. ~15% of orders are cancelled or returned — a significant operational and revenue risk.
6. UPI + Credit Card dominate payments; COD share is declining but still notable.

## 5. Recommendations
- **Launch city-targeted campaigns** for Jaipur, Lucknow, Ahmedabad to unlock tier-2 growth.
- **Investigate returns and cancellations** at the category and product level — Electronics returns are especially costly.
- **Introduce loyalty programs** to convert single-order customers into repeat buyers.
- **Bundle low-ticket categories** (Books, Beauty) with high-traffic categories (Electronics) to increase cross-sell AOV.
- **Scale festive campaigns** using the seasonal peak pattern to front-load inventory and marketing spend.

## 6. Limitations
- Dataset is synthetic — real-world patterns may differ.
- Customer demographics (age, gender) are not available for deeper segmentation.
- Product cost data is absent, so profit margins cannot be calculated.
- Delivery time data is unavailable — delay analysis is out of scope.

---
*Report prepared by: Akshat Singh | WeIntern Pvt Ltd | Data Science Internship — Week 1*
